In [ ]:
#Do we have a match in APOGEE for these 5 Gaia coordinates?
import numpy as np
import pandas as pd
from astropy.table import Table
from astropy.coordinates import SkyCoord
import astropy.units as u

# Your 5 Gaia coordinates
gaia = pd.DataFrame({
    "ra":  [126.8096397, 127.2591447, 128.4366575, 128.5937231, 126.9017987],
    "dec": [-20.79560042, -20.63431142, -19.69503396, -19.00372779, -19.20759092]
})

# Load APOGEE allStar
tab = Table.read("allStar-dr17-synspec_rev1.fits", hdu=1)

# Clean APOGEE coordinates
ap_ra = np.array(tab["RA"], dtype=float)
ap_dec = np.array(tab["DEC"], dtype=float)
valid = np.isfinite(ap_ra) & np.isfinite(ap_dec)
tab_valid = tab[valid]

# Build coordinates
gaia_coords = SkyCoord(
    ra=gaia["ra"].values * u.deg,
    dec=gaia["dec"].values * u.deg
)

apogee_coords = SkyCoord(
    ra=np.array(tab_valid["RA"], dtype=float) * u.deg,
    dec=np.array(tab_valid["DEC"], dtype=float) * u.deg
)

# Match
idx, sep2d, _ = gaia_coords.match_to_catalog_sky(apogee_coords)

# Print results
max_sep = 3.0 * u.arcmin

for i in range(len(gaia)):
    print(f"\nGaia target {i+1}")
    print(f"  Input RA,Dec = {gaia.loc[i,'ra']}, {gaia.loc[i,'dec']}")
    print(f"  Closest APOGEE sep = {sep2d[i].arcsec:.3f} arcsec")

    if sep2d[i] < max_sep:
        row = tab_valid[idx[i]]
        print(f"  APOGEE_ID = {row['APOGEE_ID']}")
        print(f"  APOGEE RA,Dec = {row['RA']}, {row['DEC']}")
        if 'FIELD' in tab_valid.colnames:
            print(f"  FIELD = {row['FIELD']}")
        if 'TELESCOPE' in tab_valid.colnames:
            print(f"  TELESCOPE = {row['TELESCOPE']}")
    else:
        print("  No secure match within 3 arcmin")

KeyboardInterrupt: 

In [ ]:
#How many APOGEE stars are within 1 degree of each of these targets?
from astropy.coordinates import SkyCoord
import astropy.units as u
import numpy as np
from astropy.table import Table

tab = Table.read("allStar-dr17-synspec_rev1.fits", hdu=1)

ap_ra = np.array(tab["RA"], dtype=float)
ap_dec = np.array(tab["DEC"], dtype=float)
valid = np.isfinite(ap_ra) & np.isfinite(ap_dec)
tab_valid = tab[valid]

apogee_coords = SkyCoord(
    ra=np.array(tab_valid["RA"], dtype=float) * u.deg,
    dec=np.array(tab_valid["DEC"], dtype=float) * u.deg
)

targets = [
    (126.8096397, -20.79560042),
    (127.2591447, -20.63431142),
    (128.4366575, -19.69503396),
    (128.5937231, -19.00372779),
    (126.9017987, -19.20759092),
]

for i, (ra, dec) in enumerate(targets, start=1):
    c = SkyCoord(ra=ra*u.deg, dec=dec*u.deg)
    sep = c.separation(apogee_coords)
    nearby = sep < (1.0 * u.deg)

    print(f"\nTarget {i}: {ra}, {dec}")
    print(f"APOGEE stars within 1 deg: {nearby.sum()}")


Target 1: 126.8096397, -20.79560042
APOGEE stars within 1 deg: 0

Target 2: 127.2591447, -20.63431142
APOGEE stars within 1 deg: 33

Target 3: 128.4366575, -19.69503396
APOGEE stars within 1 deg: 46

Target 4: 128.5937231, -19.00372779
APOGEE stars within 1 deg: 0

Target 5: 126.9017987, -19.20759092
APOGEE stars within 1 deg: 2


In [ ]:
#Do we have a match in APOGEE for these 5 Gaia coordinates? If not, what's the closest APOGEE star to each target?
import numpy as np
from astropy.table import Table
from astropy.coordinates import SkyCoord
import astropy.units as u

tab = Table.read("allStar-dr17-synspec_rev1.fits", hdu=1)

ap_ra = np.array(tab["RA"], dtype=float)
ap_dec = np.array(tab["DEC"], dtype=float)
valid = np.isfinite(ap_ra) & np.isfinite(ap_dec)
tab_valid = tab[valid]

apogee_coords = SkyCoord(
    ra=np.array(tab_valid["RA"], dtype=float) * u.deg,
    dec=np.array(tab_valid["DEC"], dtype=float) * u.deg
)

targets = [
    (126.8096397, -20.79560042),
    (127.2591447, -20.63431142),
    (128.4366575, -19.69503396),
    (128.5937231, -19.00372779),
    (126.9017987, -19.20759092),
]

for i, (ra, dec) in enumerate(targets, start=1):
    c = SkyCoord(ra=ra*u.deg, dec=dec*u.deg)
    idx, sep2d, _ = c.match_to_catalog_sky(apogee_coords)
    row = tab_valid[idx]

    print(f"\nTarget {i}")
    print(f"Input: {ra}, {dec}")
    print(f"Nearest APOGEE_ID: {row['APOGEE_ID']}")
    sep_arcmin = np.atleast_1d(sep2d.to(u.arcmin).value)[0]
    print(f"Separation: {sep_arcmin:.2f} arcmin")
    print(f"APOGEE position: {float(row['RA'])}, {float(row['DEC'])}")
    if 'FIELD' in tab_valid.colnames:
        print(f"FIELD: {row['FIELD']}")
    if 'TELESCOPE' in tab_valid.colnames:
        print(f"TELESCOPE: {row['TELESCOPE']}")


Target 1
Input: 126.8096397, -20.79560042
Nearest APOGEE_ID: 2M08313774-2044159
Separation: 61.68 arcmin
APOGEE position: 127.907263, -20.737762
FIELD: 129-21_btx
TELESCOPE: apo25m

Target 2
Input: 127.2591447, -20.63431142
Nearest APOGEE_ID: 2M08313774-2044159
Separation: 36.91 arcmin
APOGEE position: 127.907263, -20.737762
FIELD: 129-21_btx
TELESCOPE: apo25m

Target 3
Input: 128.4366575, -19.69503396
Nearest APOGEE_ID: 2M08335791-2004002
Separation: 22.51 arcmin
APOGEE position: 128.491292, -20.066736
FIELD: 129-21_btx
TELESCOPE: apo25m

Target 4
Input: 128.5937231, -19.00372779
Nearest APOGEE_ID: 2M08335791-2004002
Separation: 64.04 arcmin
APOGEE position: 128.491292, -20.066736
FIELD: 129-21_btx
TELESCOPE: apo25m

Target 5
Input: 126.9017987, -19.20759092
Nearest APOGEE_ID: 2M08271045-1813311
Separation: 59.26 arcmin
APOGEE position: 126.793544, -18.225332
FIELD: 240+12
TELESCOPE: lco25m


In [10]:
#Cross-matching code
import numpy as np
import pandas as pd
from astropy.table import Table
from astropy.coordinates import SkyCoord
import astropy.units as u

# --------------------------------------------------
# Files
# --------------------------------------------------
apogee_file = "allStar-dr17-synspec_rev1.fits"   # APOGEE DR17 allStar
gaia_file   = "gaia_targets.csv"                 # your Gaia table

# --------------------------------------------------
# User settings
# --------------------------------------------------
match_radius = 1.0 * u.arcsec

# Optional APOGEE quality cuts
min_snr = 80
require_aspcapflag_zero = True

# --------------------------------------------------
# 1. Load APOGEE observed stars
# --------------------------------------------------
ap = Table.read(apogee_file, hdu=1)

# Keep only rows with valid coordinates
ap_ra  = np.array(ap["RA"], dtype=float)
ap_dec = np.array(ap["DEC"], dtype=float)

valid = np.isfinite(ap_ra) & np.isfinite(ap_dec)
ap = ap[valid]

# Optional quality cuts
mask = np.ones(len(ap), dtype=bool)

if "SNR" in ap.colnames:
    mask &= np.array(ap["SNR"], dtype=float) >= min_snr

if require_aspcapflag_zero and "ASPCAPFLAG" in ap.colnames:
    mask &= np.array(ap["ASPCAPFLAG"]) == 0

ap = ap[mask]

print(f"APOGEE stars after cuts: {len(ap)}")

# --------------------------------------------------
# 2. Load Gaia table
#    Must have columns: ra, dec
# --------------------------------------------------
gaia = pd.read_csv(gaia_file)

gaia = gaia[np.isfinite(gaia["ra"]) & np.isfinite(gaia["dec"])].copy()
gaia = gaia.reset_index(drop=True)

print(f"Gaia stars with valid coordinates: {len(gaia)}")

# --------------------------------------------------
# 3. Build SkyCoord objects
# --------------------------------------------------
ap_coords = SkyCoord(
    ra=np.array(ap["RA"], dtype=float) * u.deg,
    dec=np.array(ap["DEC"], dtype=float) * u.deg
)

gaia_coords = SkyCoord(
    ra=gaia["ra"].values * u.deg,
    dec=gaia["dec"].values * u.deg
)

# --------------------------------------------------
# 4. Match Gaia -> APOGEE
# --------------------------------------------------
idx, sep2d, _ = gaia_coords.match_to_catalog_sky(ap_coords)
matched = sep2d < match_radius

# --------------------------------------------------
# 5. Build output table
# --------------------------------------------------
out = gaia.copy()
out["matched_to_apogee"] = matched
out["match_sep_arcsec"] = sep2d.arcsec

# Initialize APOGEE columns
out["APOGEE_ID"] = None
out["APOGEE_RA"] = np.nan
out["APOGEE_DEC"] = np.nan
out["FIELD"] = None
out["TELESCOPE"] = None
out["APOGEE_SNR"] = np.nan
out["APOGEE_TEFF"] = np.nan
out["APOGEE_LOGG"] = np.nan
out["APOGEE_M_H"] = np.nan

# Fill matched rows
m = matched

out.loc[m, "APOGEE_ID"]  = [str(x) for x in ap["APOGEE_ID"][idx[m]]]
out.loc[m, "APOGEE_RA"]  = np.array(ap["RA"][idx[m]], dtype=float)
out.loc[m, "APOGEE_DEC"] = np.array(ap["DEC"][idx[m]], dtype=float)

if "FIELD" in ap.colnames:
    out.loc[m, "FIELD"] = [str(x) for x in ap["FIELD"][idx[m]]]

if "TELESCOPE" in ap.colnames:
    out.loc[m, "TELESCOPE"] = [str(x) for x in ap["TELESCOPE"][idx[m]]]

if "SNR" in ap.colnames:
    out.loc[m, "APOGEE_SNR"] = np.array(ap["SNR"][idx[m]], dtype=float)

if "TEFF" in ap.colnames:
    out.loc[m, "APOGEE_TEFF"] = np.array(ap["TEFF"][idx[m]], dtype=float)

if "LOGG" in ap.colnames:
    out.loc[m, "APOGEE_LOGG"] = np.array(ap["LOGG"][idx[m]], dtype=float)

if "M_H" in ap.colnames:
    out.loc[m, "APOGEE_M_H"] = np.array(ap["M_H"][idx[m]], dtype=float)

# --------------------------------------------------
# 6. Save results
# --------------------------------------------------
out.to_csv("gaia_crossmatched_to_apogee.csv", index=False)

matched_only = out[out["matched_to_apogee"]].copy()
matched_only.to_csv("gaia_apogee_overlap_only.csv", index=False)

print(f"Matched {matched.sum()} / {len(out)} Gaia stars within {match_radius}")
print("Saved:")
print("  gaia_crossmatched_to_apogee.csv")
print("  gaia_apogee_overlap_only.csv")

APOGEE stars after cuts: 309891
Gaia stars with valid coordinates: 50
Matched 0 / 50 Gaia stars within 1.0 arcsec
Saved:
  gaia_crossmatched_to_apogee.csv
  gaia_apogee_overlap_only.csv


In [11]:
#Diagnostics
import numpy as np
import pandas as pd
from astropy.table import Table
from astropy.coordinates import SkyCoord
import astropy.units as u

apogee_file = "allStar-dr17-synspec_rev1.fits"
gaia_file = "gaia_targets.csv"

match_radius = 1.0 * u.arcsec

# ----------------------------
# 1. Load APOGEE with NO quality cuts
# ----------------------------
ap = Table.read(apogee_file, hdu=1)

print("Original APOGEE rows:", len(ap))

ap_ra = np.array(ap["RA"], dtype=float)
ap_dec = np.array(ap["DEC"], dtype=float)

valid_ap = np.isfinite(ap_ra) & np.isfinite(ap_dec)
ap = ap[valid_ap]

print("APOGEE rows with valid coordinates:", len(ap))

# ----------------------------
# 2. Load Gaia
# ----------------------------
gaia = pd.read_csv(gaia_file)
print("Original Gaia rows:", len(gaia))

gaia = gaia[np.isfinite(gaia["ra"]) & np.isfinite(gaia["dec"])].copy()
gaia = gaia.reset_index(drop=True)

print("Gaia rows with valid coordinates:", len(gaia))

# ----------------------------
# 3. Build coordinates
# ----------------------------
ap_coords = SkyCoord(
    ra=np.array(ap["RA"], dtype=float) * u.deg,
    dec=np.array(ap["DEC"], dtype=float) * u.deg
)

gaia_coords = SkyCoord(
    ra=gaia["ra"].values * u.deg,
    dec=gaia["dec"].values * u.deg
)

# ----------------------------
# 4. Match Gaia -> APOGEE
# ----------------------------
idx, sep2d, _ = gaia_coords.match_to_catalog_sky(ap_coords)

# convert safely to plain array
sep_arcsec = np.atleast_1d(sep2d.to(u.arcsec).value)

# ----------------------------
# 5. Diagnostics
# ----------------------------
print("\nNearest-neighbor separation stats:")
print("  min  =", np.min(sep_arcsec), "arcsec")
print("  median =", np.median(sep_arcsec), "arcsec")
print("  max  =", np.max(sep_arcsec), "arcsec")

for rad in [1, 2, 5, 10]:
    n = np.sum(sep_arcsec < rad)
    print(f"Matches within {rad} arcsec: {n}")

# ----------------------------
# 6. Save results
# ----------------------------
out = gaia.copy()
out["nearest_apogee_sep_arcsec"] = sep_arcsec
out["matched_1arcsec"] = sep_arcsec < 1.0
out["matched_2arcsec"] = sep_arcsec < 2.0
out["matched_5arcsec"] = sep_arcsec < 5.0

out["APOGEE_ID"] = [str(x) for x in ap["APOGEE_ID"][idx]]
out["APOGEE_RA"] = np.array(ap["RA"][idx], dtype=float)
out["APOGEE_DEC"] = np.array(ap["DEC"][idx], dtype=float)

if "FIELD" in ap.colnames:
    out["FIELD"] = [str(x) for x in ap["FIELD"][idx]]

if "TELESCOPE" in ap.colnames:
    out["TELESCOPE"] = [str(x) for x in ap["TELESCOPE"][idx]]

out.to_csv("gaia_to_apogee_diagnostic_matches.csv", index=False)
print("\nSaved gaia_to_apogee_diagnostic_matches.csv")

Original APOGEE rows: 733901
APOGEE rows with valid coordinates: 733900
Original Gaia rows: 50
Gaia rows with valid coordinates: 50

Nearest-neighbor separation stats:
  min  = 213.7757557407901 arcsec
  median = 5375.914011873405 arcsec
  max  = 12427.19062565728 arcsec
Matches within 1 arcsec: 0
Matches within 2 arcsec: 0
Matches within 5 arcsec: 0
Matches within 10 arcsec: 0

Saved gaia_to_apogee_diagnostic_matches.csv


In [17]:
import numpy as np
import pandas as pd
from astropy.table import Table

ap = Table.read("allStar-dr17-synspec_rev1.fits", hdu=1)

ra = np.array(ap["RA"], dtype=float)
dec = np.array(ap["DEC"], dtype=float)

valid = np.isfinite(ra) & np.isfinite(dec)
ap = ap[valid]

mask = np.ones(len(ap), dtype=bool)

# example sky-region cut
mask &= (np.array(ap["RA"], dtype=float) > 245) & (np.array(ap["RA"], dtype=float) < 255)
mask &= (np.array(ap["DEC"], dtype=float) > 25) & (np.array(ap["DEC"], dtype=float) < 35)

# example quality cuts
#if "SNR" in ap.colnames:
    #mask &= np.array(ap["SNR"], dtype=float) > 80

selected = ap[mask]

df = pd.DataFrame({
    "APOGEE_ID": [str(x) for x in selected["APOGEE_ID"]],
    "RA": np.array(selected["RA"], dtype=float),
    "DEC": np.array(selected["DEC"], dtype=float),
})

print("Selected APOGEE stars:", len(df))
print(df.head())

df.to_csv("apogee_stars_RA_Dec_cut2_smaller.csv", index=False)

Selected APOGEE stars: 4068
            APOGEE_ID          RA        DEC
0  2M16200012+2733344  245.000524  27.559576
1  2M16200058+2638119  245.002427  26.636641
2  2M16200096+3029038  245.004034  30.484415
3  2M16200130+3054529  245.005444  30.914719
4  2M16200198+2825014  245.008278  28.417061
